In [14]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [15]:
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

base_path = "/content/drive/MyDrive/Colab Notebooks/"

with open(base_path + "isolation_forest_results.pkl", "rb") as f:
    risultati_if = pickle.load(f)

with open(base_path + "lof_results.pkl", "rb") as f:
    risultati_lof = pickle.load(f)

with open(base_path + "one_class_svm_results.pkl", "rb") as f:
    risultati_ocsvm = pickle.load(f)

with open(base_path + "autoencoder_results.pkl", "rb") as f:
    risultati_ae = pickle.load(f)

print("Modelli caricati correttamente.")

print("\nIsolation Forest:")
print(risultati_if.keys())

print("\nLOF:")
print(risultati_lof.keys())

print("\nOne-Class SVM:")
print(risultati_ocsvm.keys())

print("\nAutoencoder:")
print(risultati_ae.keys())

Modelli caricati correttamente.

Isolation Forest:
dict_keys(['model', 'scaler', 'anomaly_score_train', 'anomaly_score_validation', 'anomaly_score_test', 'auc_sus', 'auc_evil', 'threshold_selection', 'feature_sensitivity', 'top10_features', 'test_feature_profiles_standardized', 'test_feature_profiles_original', 'test_results'])

LOF:
dict_keys(['model', 'scaler', 'anomaly_score_train', 'anomaly_score_validation', 'anomaly_score_test', 'auc_sus', 'auc_evil', 'thresholds_train', 'threshold_selection', 'feature_sensitivity', 'sensitivity_dataset', 'sensitivity_repeats', 'top10_features', 'test_feature_profiles', 'hyperparameters'])

One-Class SVM:
dict_keys(['model', 'scaler', 'hyperparameters', 'anomaly_score_train', 'anomaly_score_validation', 'anomaly_score_test', 'auc_sus', 'auc_evil', 'thresholds_train', 'threshold_selection', 'feature_sensitivity', 'top10_features', 'test_feature_profiles', 'sensitivity_dataset', 'sensitivity_repeats'])

Autoencoder:
dict_keys(['model', 'scaler', 'a

In [16]:
modelli = {
    "Isolation Forest": risultati_if,
    "LOF": risultati_lof,
    "One-Class SVM": risultati_ocsvm,
    "Autoencoder": risultati_ae
}

for nome, risultati in modelli.items():
    print(f"\n{nome}")
    print("TRAIN     :", len(risultati["anomaly_score_train"]))
    print("VALIDATION:", len(risultati["anomaly_score_validation"]))
    print("TEST      :", len(risultati["anomaly_score_test"]))


Isolation Forest
TRAIN     : 128
VALIDATION: 22
TEST      : 13

LOF
TRAIN     : 128
VALIDATION: 22
TEST      : 13

One-Class SVM
TRAIN     : 128
VALIDATION: 22
TEST      : 13

Autoencoder
TRAIN     : 128
VALIDATION: 22
TEST      : 13


## Normalizzazione preliminare degli anomaly score tramite Z-score

I quattro modelli utilizzati nell'ensemble — **Isolation Forest, Local Outlier Factor, One-Class SVM e Autoencoder** — producono anomaly score su scale numeriche differenti.

Di conseguenza, gli score grezzi non possono essere combinati direttamente, poiché un modello caratterizzato da valori numericamente più elevati potrebbe influenzare maggiormente l'ensemble indipendentemente dal proprio contributo relativo alla rilevazione delle anomalie.

Come analisi preliminare viene quindi considerata una **standardizzazione Z-score** applicata separatamente agli anomaly score di ciascun modello:

$$
z = \frac{s - \mu_{\text{train}}}{\sigma_{\text{train}}},
$$

dove:

* \(s\) è l'anomaly score originale;
* \(\mu_{\text{train}}\) è la media degli anomaly score sul TRAIN;
* \(\sigma_{\text{train}}\) è la deviazione standard degli anomaly score sul TRAIN.

Media e deviazione standard vengono calcolate **esclusivamente sul TRAIN**. In questa fase preliminare la trasformazione viene applicata soltanto agli score di TRAIN e VALIDATION, mantenendo il TEST escluso dalle decisioni relative alla strategia di normalizzazione dell'ensemble.

Dopo la trasformazione, gli anomaly score del TRAIN di ciascun modello presentano media circa pari a 0 e deviazione standard circa pari a 1. Gli score del VALIDATION possono invece assumere valori anche esterni all'intervallo osservato sul TRAIN, poiché vengono standardizzati utilizzando esclusivamente i parametri stimati sulla distribuzione di training.

La convenzione rimane invariata: **valori più elevati dello score normalizzato indicano un comportamento maggiormente anomalo**.

In [17]:
import numpy as np

score_norm = {}

for nome, risultati in modelli.items():

    train = np.asarray(
        risultati["anomaly_score_train"],
        dtype=float
    ).ravel()

    validation = np.asarray(
        risultati["anomaly_score_validation"],
        dtype=float
    ).ravel()

    # Parametri calcolati ESCLUSIVAMENTE sul TRAIN
    mu_train = np.mean(train)
    std_train = np.std(train)

    if std_train == 0:
        raise ValueError(
            f"Deviazione standard nulla nel TRAIN di {nome}"
        )

    score_norm[nome] = {
        "train": (train - mu_train) / std_train,
        "validation": (validation - mu_train) / std_train,
        "mu_train": mu_train,
        "std_train": std_train
    }

    print(f"\n{nome}")
    print(f"Media TRAIN originale: {mu_train:.6f}")
    print(f"Std TRAIN originale  : {std_train:.6f}")
    print(
        "Media TRAIN normalizzato:",
        round(np.mean(score_norm[nome]["train"]), 6)
    )
    print(
        "Std TRAIN normalizzato  :",
        round(np.std(score_norm[nome]["train"]), 6)
    )


Isolation Forest
Media TRAIN originale: -0.085906
Std TRAIN originale  : 0.064499
Media TRAIN normalizzato: 0.0
Std TRAIN normalizzato  : 1.0

LOF
Media TRAIN originale: 1.879450
Std TRAIN originale  : 1.488799
Media TRAIN normalizzato: -0.0
Std TRAIN normalizzato  : 1.0

One-Class SVM
Media TRAIN originale: -0.162734
Std TRAIN originale  : 0.376481
Media TRAIN normalizzato: -0.0
Std TRAIN normalizzato  : 1.0

Autoencoder
Media TRAIN originale: 0.091916
Std TRAIN originale  : 0.188485
Media TRAIN normalizzato: -0.0
Std TRAIN normalizzato  : 1.0


## Costruzione preliminare dell'ensemble mediante Z-score

Dopo la standardizzazione preliminare degli anomaly score dei quattro modelli, viene costruito un ensemble mediante **media aritmetica non pesata** degli score Z-score prodotti da:

* Isolation Forest;
* Local Outlier Factor;
* One-Class SVM;
* Autoencoder.

Per ogni finestra temporale, lo score preliminare dell'ensemble è definito come:

$$
s_{\text{ensemble}}
=
\frac{
z_{\text{IF}} +
z_{\text{LOF}} +
z_{\text{OCSVM}} +
z_{\text{AE}}
}{4}.
$$

Ogni modello contribuisce quindi allo score complessivo con lo stesso peso, pari al **25\%**.

Questa configurazione viene utilizzata esclusivamente come analisi preliminare della combinazione degli score. L'ensemble Z-score viene calcolato soltanto su TRAIN e VALIDATION, in modo da verificare il comportamento della normalizzazione senza utilizzare il TEST per effettuare scelte di configurazione.

La convenzione rimane invariata: **valori più elevati dell'ensemble score indicano finestre considerate maggiormente anomale dall'insieme dei modelli**.

In [18]:
ensemble_score_train = np.mean(
    np.vstack([
        score_norm["Isolation Forest"]["train"],
        score_norm["LOF"]["train"],
        score_norm["One-Class SVM"]["train"],
        score_norm["Autoencoder"]["train"]
    ]),
    axis=0
)

ensemble_score_validation = np.mean(
    np.vstack([
        score_norm["Isolation Forest"]["validation"],
        score_norm["LOF"]["validation"],
        score_norm["One-Class SVM"]["validation"],
        score_norm["Autoencoder"]["validation"]
    ]),
    axis=0
)

print("Ensemble Z-score preliminare costruito correttamente.")

print("\nDimensioni:")
print("TRAIN     :", ensemble_score_train.shape)
print("VALIDATION:", ensemble_score_validation.shape)

print("\nStatistiche ensemble score:")
print(
    "TRAIN      ->",
    f"min={ensemble_score_train.min():.4f}",
    f"max={ensemble_score_train.max():.4f}",
    f"mean={ensemble_score_train.mean():.4f}"
)

print(
    "VALIDATION ->",
    f"min={ensemble_score_validation.min():.4f}",
    f"max={ensemble_score_validation.max():.4f}",
    f"mean={ensemble_score_validation.mean():.4f}"
)

Ensemble Z-score preliminare costruito correttamente.

Dimensioni:
TRAIN     : (128,)
VALIDATION: (22,)

Statistiche ensemble score:
TRAIN      -> min=-0.7383 max=3.8008 mean=-0.0000
VALIDATION -> min=-0.4147 max=10.4413 mean=1.0224


## Controllo degli score normalizzati estremi sul validation set

Prima di definire la strategia definitiva di normalizzazione dell'ensemble, viene verificata sul VALIDATION la presenza di valori elevati negli anomaly score standardizzati.

La standardizzazione Z-score utilizza media e deviazione standard calcolate esclusivamente sul TRAIN. Di conseguenza, uno score del VALIDATION molto distante dalla distribuzione osservata durante il training può assumere un valore standardizzato particolarmente elevato.

Poiché l'ensemble è costruito mediante una media aritmetica non pesata, uno Z-score molto elevato prodotto da un singolo modello può influenzare in modo dominante lo score complessivo, riducendo la comparabilità dei contributi dei quattro detector.

Il controllo viene quindi effettuato utilizzando esclusivamente TRAIN e VALIDATION. Il TEST rimane escluso da questa analisi e non viene utilizzato per scegliere la strategia definitiva di normalizzazione.

In [19]:
print("Range degli Z-score sul VALIDATION:\n")

for nome in modelli.keys():
    z_validation = score_norm[nome]["validation"]

    print(nome)
    print(f"  min : {np.min(z_validation):.4f}")
    print(f"  max : {np.max(z_validation):.4f}")
    print(f"  mean: {np.mean(z_validation):.4f}")

    indice_max = np.argmax(z_validation)

    print(f"  finestra con Z massimo: {indice_max}")
    print(f"  Z massimo             : {z_validation[indice_max]:.4f}")
    print()

Range degli Z-score sul VALIDATION:

Isolation Forest
  min : -0.8728
  max : 1.9937
  mean: 0.4510
  finestra con Z massimo: 0
  Z massimo             : 1.9937

LOF
  min : -0.5863
  max : 2.2224
  mean: 0.6414
  finestra con Z massimo: 20
  Z massimo             : 2.2224

One-Class SVM
  min : -0.8289
  max : 6.3605
  mean: 1.0351
  finestra con Z massimo: 10
  Z massimo             : 6.3605

Autoencoder
  min : -0.4673
  max : 32.0271
  mean: 1.9620
  finestra con Z massimo: 10
  Z massimo             : 32.0271



In [20]:
indice_max_ensemble = np.argmax(ensemble_score_validation)

print(
    "Finestra VALIDATION con ensemble score massimo:",
    indice_max_ensemble
)

print(
    "Ensemble score:",
    ensemble_score_validation[indice_max_ensemble]
)

print("\nContributo dei quattro modelli:")

for nome in modelli.keys():
    print(
        f"{nome:20s}: "
        f"{score_norm[nome]['validation'][indice_max_ensemble]:.4f}"
    )

Finestra VALIDATION con ensemble score massimo: 10
Ensemble score: 10.441316958926794

Contributo dei quattro modelli:
Isolation Forest    : 1.7517
LOF                 : 1.6260
One-Class SVM       : 6.3605
Autoencoder         : 32.0271


## Normalizzazione degli anomaly score mediante percentile empirico

L'analisi preliminare condotta su TRAIN e VALIDATION ha evidenziato che la standardizzazione Z-score può produrre valori particolarmente elevati rispetto alla distribuzione osservata durante il TRAIN. Nel VALIDATION questo comportamento risulta soprattutto evidente per l'Autoencoder e può influenzare in modo rilevante la media utilizzata per costruire l'ensemble.

Poiché l'ensemble combina i quattro detector mediante una media aritmetica non pesata, uno Z-score particolarmente elevato prodotto da un singolo modello potrebbe dominare lo score complessivo e ridurre la comparabilità dei contributi dei diversi detector.

Per rendere più robusta la combinazione degli anomaly score viene quindi adottata come strategia definitiva una **normalizzazione basata sul percentile empirico rispetto alla distribuzione del TRAIN**.

Per ciascun modello, ogni anomaly score viene trasformato nella proporzione di score del TRAIN minori o uguali al valore considerato.

Il valore normalizzato appartiene quindi all'intervallo \([0,1]\):

* valori vicini a 0 indicano score bassi rispetto alla distribuzione del TRAIN;
* valori vicini a 1 indicano score particolarmente elevati rispetto alla distribuzione del TRAIN.

La distribuzione di riferimento viene costruita **esclusivamente utilizzando il TRAIN**. La strategia di normalizzazione viene definita sulla base dell'analisi di TRAIN e VALIDATION; successivamente la stessa trasformazione, senza ulteriori modifiche, viene applicata anche al TEST per la valutazione finale.

Questa trasformazione preserva l'ordinamento degli anomaly score di ciascun modello e limita l'effetto delle differenti scale numeriche e dei valori estremi nella successiva combinazione mediante media aritmetica.

In [21]:
def train_percentile_score(train_scores, target_scores):
    """
    Trasforma gli anomaly score in percentili empirici
    rispetto esclusivamente alla distribuzione del TRAIN.
    """

    train_scores = np.asarray(train_scores, dtype=float).ravel()
    target_scores = np.asarray(target_scores, dtype=float).ravel()

    train_sorted = np.sort(train_scores)

    percentiles = np.searchsorted(
        train_sorted,
        target_scores,
        side="right"
    ) / len(train_sorted)

    return percentiles


score_percentile = {}

for nome, risultati in modelli.items():

    train = risultati["anomaly_score_train"]
    validation = risultati["anomaly_score_validation"]
    test = risultati["anomaly_score_test"]

    score_percentile[nome] = {
        "train": train_percentile_score(train, train),
        "validation": train_percentile_score(train, validation),
        "test": train_percentile_score(train, test)
    }

    print(f"\n{nome}")
    print(
        "TRAIN      ->",
        f"min={score_percentile[nome]['train'].min():.4f}",
        f"max={score_percentile[nome]['train'].max():.4f}"
    )
    print(
        "VALIDATION ->",
        f"min={score_percentile[nome]['validation'].min():.4f}",
        f"max={score_percentile[nome]['validation'].max():.4f}"
    )
    print(
        "TEST       ->",
        f"min={score_percentile[nome]['test'].min():.4f}",
        f"max={score_percentile[nome]['test'].max():.4f}"
    )


Isolation Forest
TRAIN      -> min=0.0078 max=1.0000
VALIDATION -> min=0.2109 max=0.9531
TEST       -> min=0.2188 max=1.0000

LOF
TRAIN      -> min=0.0078 max=1.0000
VALIDATION -> min=0.3125 max=0.9375
TEST       -> min=0.0938 max=1.0000

One-Class SVM
TRAIN      -> min=0.0078 max=1.0000
VALIDATION -> min=0.1797 max=1.0000
TEST       -> min=0.4297 max=1.0000

Autoencoder
TRAIN      -> min=0.0078 max=1.0000
VALIDATION -> min=0.1562 max=1.0000
TEST       -> min=0.1797 max=1.0000


## Costruzione dell'ensemble basato sui percentili empirici

Dopo aver trasformato gli anomaly score dei singoli modelli in percentili empirici rispetto alle rispettive distribuzioni di TRAIN, viene costruito lo score finale dell'ensemble.

Per ogni finestra temporale vengono combinati gli score normalizzati di:

* Isolation Forest;
* Local Outlier Factor;
* One-Class SVM;
* Autoencoder.

L'ensemble utilizza una **media aritmetica non pesata**, assegnando quindi lo stesso peso, pari al 25%, a ciascun modello:

$$
s_{\text{ensemble}}
=
\frac{
p_{\text{IF}} +
p_{\text{LOF}} +
p_{\text{OCSVM}} +
p_{\text{AE}}
}{4}
$$

dove \(p\) rappresenta il percentile empirico dell'anomaly score del rispettivo modello rispetto alla sua distribuzione di TRAIN.

Lo score finale dell'ensemble è compreso tra 0 e 1. Valori elevati indicano che, complessivamente, i modelli assegnano alla finestra anomaly score elevati rispetto a quelli osservati durante il TRAIN.

La media non pesata permette inoltre di attribuire lo stesso contributo ai quattro detector, evitando che le differenti scale numeriche degli anomaly score originali influenzino artificialmente il risultato.


In [22]:
# Ensemble definitivo basato sui percentili empirici

ensemble_score_train = np.mean(
    np.vstack([
        score_percentile["Isolation Forest"]["train"],
        score_percentile["LOF"]["train"],
        score_percentile["One-Class SVM"]["train"],
        score_percentile["Autoencoder"]["train"]
    ]),
    axis=0
)

ensemble_score_validation = np.mean(
    np.vstack([
        score_percentile["Isolation Forest"]["validation"],
        score_percentile["LOF"]["validation"],
        score_percentile["One-Class SVM"]["validation"],
        score_percentile["Autoencoder"]["validation"]
    ]),
    axis=0
)

ensemble_score_test = np.mean(
    np.vstack([
        score_percentile["Isolation Forest"]["test"],
        score_percentile["LOF"]["test"],
        score_percentile["One-Class SVM"]["test"],
        score_percentile["Autoencoder"]["test"]
    ]),
    axis=0
)

print("Ensemble percentile costruito correttamente.")

print("\nDimensioni:")
print("TRAIN     :", ensemble_score_train.shape)
print("VALIDATION:", ensemble_score_validation.shape)
print("TEST      :", ensemble_score_test.shape)

print("\nRange ensemble score:")
print(
    f"TRAIN      -> min={ensemble_score_train.min():.4f}, "
    f"max={ensemble_score_train.max():.4f}"
)
print(
    f"VALIDATION -> min={ensemble_score_validation.min():.4f}, "
    f"max={ensemble_score_validation.max():.4f}"
)
print(
    f"TEST       -> min={ensemble_score_test.min():.4f}, "
    f"max={ensemble_score_test.max():.4f}"
)

Ensemble percentile costruito correttamente.

Dimensioni:
TRAIN     : (128,)
VALIDATION: (22,)
TEST      : (13,)

Range ensemble score:
TRAIN      -> min=0.1309, max=0.9668
VALIDATION -> min=0.3418, max=0.9551
TEST       -> min=0.3770, max=1.0000


## Label di VALIDATION per la selezione della soglia

Dopo aver costruito l'ensemble anomaly score, è necessario determinare una soglia che permetta di trasformare lo score continuo in una classificazione binaria delle finestre.

La selezione della soglia viene effettuata utilizzando il dataset di **VALIDATION**, evitando di utilizzare il TEST durante questa fase.

In particolare, vengono recuperate le label `sus_window` e `evil_window` associate alle finestre temporali.

La label `sus_window` indica la presenza di almeno un evento SUS all'interno della finestra, mentre `evil_window` indica la presenza di almeno un evento EVIL.

La VALIDATION verrà utilizzata per valutare e selezionare la soglia relativa alle finestre SUS. La possibilità di selezionare una soglia EVIL dipende invece dalla presenza di finestre EVIL positive nella VALIDATION.

Il TEST verrà utilizzato esclusivamente dopo aver completato e fissato le scelte relative all'ensemble e alla soglia.


In [23]:
# Caricamento delle label preparate nel preprocessing

with open(base_path + "beth_prepared.pkl", "rb") as f:
    dati_preparati = pickle.load(f)

y_validation = dati_preparati["y_validation"]
y_test = dati_preparati["y_test"]

# Label binarie
y_val_sus = y_validation["sus_window"].astype(int).values
y_val_evil = y_validation["evil_window"].astype(int).values

y_test_sus = y_test["sus_window"].astype(int).values
y_test_evil = y_test["evil_window"].astype(int).values

# Controllo fondamentale dell'allineamento
assert len(y_val_sus) == len(ensemble_score_validation), \
    "ERRORE: label e score VALIDATION non allineati"

assert len(y_test_sus) == len(ensemble_score_test), \
    "ERRORE: label e score TEST non allineati"

print("VALIDATION")
print("Finestre totali:", len(y_validation))
print("Finestre SUS    :", y_val_sus.sum())
print("Finestre EVIL   :", y_val_evil.sum())

print("\nTEST")
print("Finestre totali:", len(y_test))
print("Finestre SUS    :", y_test_sus.sum())
print("Finestre EVIL   :", y_test_evil.sum())

print("\nOK: label e ensemble score correttamente allineati.")

VALIDATION
Finestre totali: 22
Finestre SUS    : 2
Finestre EVIL   : 0

TEST
Finestre totali: 13
Finestre SUS    : 7
Finestre EVIL   : 5

OK: label e ensemble score correttamente allineati.


## Selezione della soglia SUS dell'ensemble

Per trasformare l'ensemble anomaly score in una classificazione binaria delle finestre è necessario definire una soglia.

Le soglie candidate vengono ricavate **esclusivamente dalla distribuzione degli ensemble score del TRAIN**, considerando i percentili:

* 90° percentile;
* 95° percentile;
* 97° percentile;
* 99° percentile.

Per ciascuna soglia, una finestra viene classificata come anomala quando il suo ensemble score è maggiore o uguale alla soglia considerata.

Le soglie candidate vengono successivamente valutate sulla **VALIDATION** rispetto alla label `sus_window`, calcolando precision, recall e F1-score.

La soglia SUS viene selezionata in base al miglior F1-score ottenuto sulla VALIDATION. In caso di parità, viene privilegiata la soluzione con recall maggiore e successivamente quella con precision maggiore.

Questa procedura mantiene separato il TEST dal processo di selezione della soglia. Il TEST verrà utilizzato solamente dopo aver fissato la configurazione finale dell'ensemble.

La VALIDATION contiene soltanto 2 finestre SUS positive; di conseguenza, le metriche ottenute devono essere interpretate con cautela, poiché la selezione della soglia si basa su un numero molto ridotto di esempi positivi.

Non viene invece selezionata una soglia specifica per EVIL, poiché la VALIDATION non contiene finestre `evil_window` positive.


In [31]:
from sklearn.metrics import precision_score, recall_score, f1_score

# Percentili del TRAIN utilizzati come soglie candidate
percentili = [90, 95, 97, 99]

thresholds_ensemble = {
    p: np.percentile(ensemble_score_train, p)
    for p in percentili
}

risultati_soglie_validation = []

for p, soglia in thresholds_ensemble.items():

    # Predizione sulla VALIDATION
    pred_val_sus = (
        ensemble_score_validation >= soglia
    ).astype(int)

    precision = precision_score(
        y_val_sus,
        pred_val_sus,
        zero_division=0
    )

    recall = recall_score(
        y_val_sus,
        pred_val_sus,
        zero_division=0
    )

    f1 = f1_score(
        y_val_sus,
        pred_val_sus,
        zero_division=0
    )

    risultati_soglie_validation.append({
        "percentile_train": p,
        "threshold": soglia,
        "n_predetti_sus": int(pred_val_sus.sum()),
        "precision": precision,
        "recall": recall,
        "f1": f1
    })

validation_threshold_df = pd.DataFrame(
    risultati_soglie_validation
)

display(validation_threshold_df)

,percentile_train,threshold,n_predetti_sus,precision,recall,f1
0,90,0.865820,6,0.166667,0.5,0.25
1,95,0.895801,3,0.000000,0.0,0.00
2,97,0.922246,3,0.000000,0.0,0.00
3,99,0.954551,1,0.000000,0.0,0.00


## Selezione finale della soglia SUS

Tra le soglie candidate ottenute dai percentili 90, 95, 97 e 99 della distribuzione degli ensemble score sul TRAIN, la soglia corrispondente al **90° percentile** ha ottenuto il miglior risultato sulla VALIDATION.

In particolare, con questa soglia:

* vengono classificate come SUS 6 finestre della VALIDATION;
* 1 delle 2 finestre realmente SUS viene individuata;
* la precision è pari a circa 0,167;
* il recall è pari a 0,50;
* l'F1-score è pari a 0,25.

Le soglie corrispondenti ai percentili 95, 97 e 99 non individuano invece nessuna delle finestre SUS presenti nella VALIDATION e producono quindi F1-score pari a 0.

La soglia SUS dell'ensemble viene pertanto fissata al **90° percentile degli ensemble score del TRAIN**.

La VALIDATION contiene solamente 2 finestre SUS positive; di conseguenza, questa selezione deve essere interpretata con cautela, poiché una singola finestra classificata correttamente o erroneamente produce variazioni rilevanti nelle metriche.

Non viene selezionata una soglia specifica per EVIL, poiché la VALIDATION non contiene finestre EVIL positive.

Una volta fissata questa configurazione, il TEST può essere utilizzato per la valutazione finale senza modificare ulteriormente la soglia sulla base dei risultati ottenuti.


In [32]:
# Selezione definitiva della soglia SUS sulla VALIDATION

best_row = validation_threshold_df.sort_values(
    by=[
        "f1",
        "recall",
        "precision",
        "percentile_train"
    ],
    ascending=[
        False,  # F1 maggiore
        False,  # a parità, recall maggiore
        False,  # poi precision maggiore
        True    # infine percentile più basso
    ]
).iloc[0]

soglia_sus_ensemble = float(best_row["threshold"])
percentile_sus_ensemble = int(best_row["percentile_train"])

print("Soglia SUS selezionata")
print("----------------------")
print(
    "Percentile TRAIN:",
    percentile_sus_ensemble
)
print(
    "Threshold:",
    round(soglia_sus_ensemble, 6)
)
print(
    "Precision VALIDATION:",
    round(best_row["precision"], 4)
)
print(
    "Recall VALIDATION:",
    round(best_row["recall"], 4)
)
print(
    "F1 VALIDATION:",
    round(best_row["f1"], 4)
)

# Nessuna soglia EVIL selezionabile sulla validation
soglia_evil_ensemble = None

Soglia SUS selezionata
----------------------
Percentile TRAIN: 90
Threshold: 0.86582
Precision VALIDATION: 0.1667
Recall VALIDATION: 0.5
F1 VALIDATION: 0.25


## Valutazione finale dell'ensemble sul TEST

Dopo aver fissato la soglia SUS utilizzando esclusivamente TRAIN e VALIDATION, l'ensemble viene valutato sul dataset di TEST.

Per la classificazione SUS viene applicata direttamente la soglia selezionata sulla VALIDATION, senza effettuare ulteriori modifiche in funzione dei risultati del TEST.

Sul TEST vengono calcolate:

* precision;
* recall;
* F1-score;
* matrice di confusione;
* ROC-AUC per la label SUS;
* ROC-AUC per la label EVIL.

La ROC-AUC misura la capacità dell'ensemble score di ordinare le finestre positive sopra quelle negative indipendentemente da una specifica soglia.

Per EVIL viene calcolata solamente la ROC-AUC, poiché non è stata selezionata una soglia EVIL sulla VALIDATION: quest'ultima non contiene infatti finestre EVIL positive.

In questo modo il TEST viene utilizzato esclusivamente per la valutazione finale e non per la selezione dei parametri o delle soglie dell'ensemble.


In [33]:
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    roc_auc_score
)

# Predizione SUS sul TEST usando la soglia già selezionata
pred_test_sus = (
    ensemble_score_test >= soglia_sus_ensemble
).astype(int)

# Metriche di classificazione SUS
precision_test_sus = precision_score(
    y_test_sus,
    pred_test_sus,
    zero_division=0
)

recall_test_sus = recall_score(
    y_test_sus,
    pred_test_sus,
    zero_division=0
)

f1_test_sus = f1_score(
    y_test_sus,
    pred_test_sus,
    zero_division=0
)

cm_test_sus = confusion_matrix(
    y_test_sus,
    pred_test_sus
)

# ROC-AUC sul TEST
auc_test_sus = roc_auc_score(
    y_test_sus,
    ensemble_score_test
)

auc_test_evil = roc_auc_score(
    y_test_evil,
    ensemble_score_test
)

print("VALUTAZIONE FINALE ENSEMBLE - TEST")
print("----------------------------------")

print("\nSUS")
print("Threshold :", round(soglia_sus_ensemble, 6))
print("Precision :", round(precision_test_sus, 4))
print("Recall    :", round(recall_test_sus, 4))
print("F1-score  :", round(f1_test_sus, 4))
print("ROC-AUC   :", round(auc_test_sus, 4))

print("\nMatrice di confusione SUS:")
print(cm_test_sus)

print("\nEVIL")
print("ROC-AUC   :", round(auc_test_evil, 4))
print("Threshold : non selezionata")

VALUTAZIONE FINALE ENSEMBLE - TEST
----------------------------------

SUS
Threshold : 0.86582
Precision : 0.625
Recall    : 0.7143
F1-score  : 0.6667
ROC-AUC   : 0.6786

Matrice di confusione SUS:
[[3 3]
 [2 5]]

EVIL
ROC-AUC   : 0.8875
Threshold : non selezionata


## Risultati finali sul TEST

La configurazione finale dell'ensemble, definita senza utilizzare il TEST per la selezione della soglia, viene valutata sulle 13 finestre del dataset di TEST.

Per la rilevazione delle finestre SUS viene utilizzata la soglia pari a **0,86582**, corrispondente al 90° percentile degli ensemble score del TRAIN e selezionata sulla VALIDATION.

Sul TEST, l'ensemble ottiene:

* **Precision SUS:** 0,625;
* **Recall SUS:** 0,7143;
* **F1-score SUS:** 0,6667;
* **ROC-AUC SUS:** 0,6786.

La matrice di confusione mostra 5 veri positivi, 3 falsi positivi, 3 veri negativi e 2 falsi negativi. L'ensemble identifica quindi correttamente **5 delle 7 finestre SUS** presenti nel TEST.

Per la label EVIL si ottiene una **ROC-AUC pari a 0,8875**, indicando una buona capacità dell'ensemble score di assegnare valori più elevati alle finestre EVIL rispetto alle finestre non-EVIL.

Non vengono calcolate metriche di classificazione threshold-based per EVIL, poiché la VALIDATION non contiene esempi EVIL positivi e non è quindi possibile selezionare una soglia EVIL indipendentemente dal TEST.

I risultati devono essere interpretati considerando la dimensione ridotta dei dataset di VALIDATION e TEST, che rende le metriche sensibili alla classificazione delle singole finestre.


## Confronto tra i modelli individuali e l'ensemble

Dopo aver completato la valutazione finale dell'ensemble, le sue prestazioni vengono confrontate con quelle dei quattro anomaly detector individuali:

* Isolation Forest;
* Local Outlier Factor;
* One-Class SVM;
* Autoencoder.

Il confronto viene effettuato utilizzando la **ROC-AUC sul TEST** per le label SUS ed EVIL.

La ROC-AUC è particolarmente adatta a questo confronto perché valuta la capacità degli anomaly score di ordinare le finestre positive rispetto a quelle negative indipendentemente dalla scelta di una specifica soglia.

Per l'ensemble vengono utilizzati i risultati ottenuti con lo score basato sulla media non pesata dei percentili empirici dei quattro modelli.

Questo confronto permette di verificare se la combinazione delle informazioni prodotte dai diversi anomaly detector migliora la capacità discriminante rispetto ai singoli modelli.


In [27]:
# Confronto finale delle ROC-AUC sul TEST

confronto_auc = pd.DataFrame({
    "Modello": [
        "Isolation Forest",
        "LOF",
        "One-Class SVM",
        "Autoencoder",
        "Ensemble"
    ],

    "AUC SUS": [
        risultati_if["auc_sus"],
        risultati_lof["auc_sus"],
        risultati_ocsvm["auc_sus"],
        risultati_ae["auc_sus"],
        auc_test_sus
    ],

    "AUC EVIL": [
        risultati_if["auc_evil"],
        risultati_lof["auc_evil"],
        risultati_ocsvm["auc_evil"],
        risultati_ae["auc_evil"],
        auc_test_evil
    ]
})

confronto_auc = confronto_auc.round(4)

display(confronto_auc)

,Modello,AUC SUS,AUC EVIL
0,Isolation Forest,0.6429,0.8250
1,LOF,0.5476,0.8250
2,One-Class SVM,0.6905,0.9000
3,Autoencoder,0.6667,0.8750
4,Ensemble,0.6786,0.8875


## Interpretazione del confronto finale

Il confronto delle ROC-AUC sul TEST mostra che l'ensemble ottiene prestazioni competitive rispetto ai quattro anomaly detector individuali, senza tuttavia risultare il metodo migliore in termini assoluti.

Per la rilevazione delle finestre **SUS**, l'ensemble raggiunge una ROC-AUC pari a **0,6786**. Tale risultato è superiore a quelli ottenuti da Isolation Forest (0,6429) e LOF (0,5476), ed è leggermente superiore a quello dell'Autoencoder (0,6667). One-Class SVM ottiene invece il valore più elevato, pari a **0,6905**.

Per le finestre **EVIL**, l'ensemble raggiunge una ROC-AUC pari a **0,8875**, superiore ai risultati di Isolation Forest e LOF (entrambi 0,8250) e dell'Autoencoder (0,8750). Anche in questo caso One-Class SVM presenta la ROC-AUC più elevata, pari a **0,9000**.

L'ensemble mostra quindi il vantaggio di combinare le informazioni provenienti da detector differenti e raggiunge risultati relativamente stabili su entrambe le tipologie di anomalie, ma non produce un miglioramento rispetto al miglior modello individuale, rappresentato in questo caso da One-Class SVM.

Non vengono effettuate ulteriori modifiche ai pesi o alla composizione dell'ensemble sulla base dei risultati del TEST, poiché ciò comporterebbe l'utilizzo del dataset di TEST per l'ottimizzazione del metodo.

I risultati devono inoltre essere interpretati considerando il numero limitato di finestre presenti nel TEST, pari a 13, che rende le metriche sensibili alla classificazione e all'ordinamento di singole osservazioni.


## Salvataggio dei risultati dell'ensemble

La configurazione finale dell'ensemble e i relativi risultati vengono salvati per consentire successive analisi e confronti senza dover ripetere l'intera procedura.

Vengono memorizzati:

* i quattro modelli che compongono l'ensemble;
* il metodo di normalizzazione degli anomaly score;
* il metodo di combinazione degli score;
* gli ensemble score di TRAIN, VALIDATION e TEST;
* le soglie candidate derivate dal TRAIN;
* la soglia SUS selezionata sulla VALIDATION;
* le metriche ottenute sulla VALIDATION durante la selezione della soglia;
* le metriche finali SUS sul TEST;
* le ROC-AUC SUS ed EVIL;
* la matrice di confusione SUS.

Non viene memorizzata una soglia EVIL operativa, poiché la VALIDATION non contiene finestre EVIL positive.


In [28]:
ensemble_results = {
    "members": [
        "Isolation Forest",
        "LOF",
        "One-Class SVM",
        "Autoencoder"
    ],

    "normalization": "empirical_train_percentile",
    "combination": "unweighted_mean",

    "anomaly_score_train": ensemble_score_train,
    "anomaly_score_validation": ensemble_score_validation,
    "anomaly_score_test": ensemble_score_test,

    "thresholds_train": thresholds_ensemble,

    "threshold_selection": {
        "SUS": soglia_sus_ensemble,
        "SUS_percentile": percentile_sus_ensemble,
        "SUS_reason": (
            "Soglia derivata dal TRAIN e selezionata sulla "
            "VALIDATION in base al miglior F1 SUS."
        ),
        "EVIL": None,
        "EVIL_percentile": None,
        "EVIL_reason": (
            "La validation non contiene finestre EVIL positive, "
            "quindi non è possibile selezionare una soglia EVIL "
            "tramite metriche di classificazione."
        )
    },

    "validation_threshold_metrics": validation_threshold_df,

    "auc_sus": auc_test_sus,
    "auc_evil": auc_test_evil,

    "test_results": {
        "SUS": {
            "precision": precision_test_sus,
            "recall": recall_test_sus,
            "f1": f1_test_sus,
            "confusion_matrix": cm_test_sus
        },
        "EVIL": {
            "threshold": None
        }
    },

    "comparison_auc": confronto_auc
}

percorso_ensemble = base_path + "ensemble_results.pkl"

with open(percorso_ensemble, "wb") as f:
    pickle.dump(ensemble_results, f)

print("Risultati ensemble salvati in:")
print(percorso_ensemble)

print("\nChiavi salvate:")
print(ensemble_results.keys())

Risultati ensemble salvati in:
/content/drive/MyDrive/Colab Notebooks/ensemble_results.pkl

Chiavi salvate:
dict_keys(['members', 'normalization', 'combination', 'anomaly_score_train', 'anomaly_score_validation', 'anomaly_score_test', 'thresholds_train', 'threshold_selection', 'validation_threshold_metrics', 'auc_sus', 'auc_evil', 'test_results', 'comparison_auc'])


## Verifica del salvataggio dell'ensemble

Come controllo finale, il file `ensemble_results.pkl` viene ricaricato da Google Drive.

La verifica consente di controllare che il file sia leggibile e che contenga correttamente la configurazione dell'ensemble, gli anomaly score, la soglia SUS selezionata e le metriche finali ottenute sul TEST.

Questo passaggio non modifica né ricalcola i risultati: verifica esclusivamente il corretto salvataggio dell'output finale.


In [29]:
# Verifica del file ensemble_results.pkl

with open(percorso_ensemble, "rb") as f:
    ensemble_verifica = pickle.load(f)

print("File ricaricato correttamente.")

print("\nChiavi presenti:")
print(ensemble_verifica.keys())

print("\nConfigurazione:")
print("Normalizzazione:", ensemble_verifica["normalization"])
print("Combinazione   :", ensemble_verifica["combination"])

print("\nSoglia SUS:")
print(
    "Percentile TRAIN:",
    ensemble_verifica["threshold_selection"]["SUS_percentile"]
)
print(
    "Threshold:",
    round(ensemble_verifica["threshold_selection"]["SUS"], 6)
)

print("\nRisultati finali TEST:")
print(
    "AUC SUS :",
    round(ensemble_verifica["auc_sus"], 4)
)
print(
    "AUC EVIL:",
    round(ensemble_verifica["auc_evil"], 4)
)
print(
    "F1 SUS  :",
    round(ensemble_verifica["test_results"]["SUS"]["f1"], 4)
)

print("\nDimensioni anomaly score:")
print(
    "TRAIN     :",
    len(ensemble_verifica["anomaly_score_train"])
)
print(
    "VALIDATION:",
    len(ensemble_verifica["anomaly_score_validation"])
)
print(
    "TEST      :",
    len(ensemble_verifica["anomaly_score_test"])
)

File ricaricato correttamente.

Chiavi presenti:
dict_keys(['members', 'normalization', 'combination', 'anomaly_score_train', 'anomaly_score_validation', 'anomaly_score_test', 'thresholds_train', 'threshold_selection', 'validation_threshold_metrics', 'auc_sus', 'auc_evil', 'test_results', 'comparison_auc'])

Configurazione:
Normalizzazione: empirical_train_percentile
Combinazione   : unweighted_mean

Soglia SUS:
Percentile TRAIN: 90
Threshold: 0.86582

Risultati finali TEST:
AUC SUS : 0.6786
AUC EVIL: 0.8875
F1 SUS  : 0.6667

Dimensioni anomaly score:
TRAIN     : 128
VALIDATION: 22
TEST      : 13


## Completamento delle informazioni salvate

Per rendere il file dell'ensemble maggiormente autosufficiente e riproducibile, vengono salvate anche alcune informazioni necessarie a ricostruire esattamente la procedura di combinazione.

In particolare vengono memorizzati:

* i pesi assegnati ai quattro anomaly detector, pari a 0,25 ciascuno;
* gli anomaly score del TRAIN dei singoli modelli, utilizzati come distribuzioni di riferimento per la trasformazione in percentili empirici.

Queste informazioni permettono di documentare in modo completo sia la normalizzazione sia la successiva combinazione degli score.


In [30]:
# Informazioni aggiuntive per rendere il file ensemble autosufficiente

ensemble_results["weights"] = {
    "Isolation Forest": 0.25,
    "LOF": 0.25,
    "One-Class SVM": 0.25,
    "Autoencoder": 0.25
}

ensemble_results["normalization_reference_train_scores"] = {
    nome: np.asarray(
        risultati["anomaly_score_train"],
        dtype=float
    ).ravel()
    for nome, risultati in modelli.items()
}

# Sovrascrittura del file con la versione completa
with open(percorso_ensemble, "wb") as f:
    pickle.dump(ensemble_results, f)

print("File ensemble aggiornato correttamente.")

print("\nPesi:")
print(ensemble_results["weights"])

print("\nRiferimenti TRAIN salvati:")
for nome, scores in ensemble_results[
    "normalization_reference_train_scores"
].items():
    print(nome, ":", len(scores), "score")

File ensemble aggiornato correttamente.

Pesi:
{'Isolation Forest': 0.25, 'LOF': 0.25, 'One-Class SVM': 0.25, 'Autoencoder': 0.25}

Riferimenti TRAIN salvati:
Isolation Forest : 128 score
LOF : 128 score
One-Class SVM : 128 score
Autoencoder : 128 score
